##Cell 1 — Config + imports

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

BASE_PATH = "/content/drive/MyDrive/Transcripts_CSS"

Mounted at /content/drive


In [ ]:
import os
import json
import re
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

BASE_PATH = "/content/drive/MyDrive/Transcripts_CSS"

OUTPUT_DIR = "/content/drive/MyDrive/Transcripts_CSS/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)


COMMUNITIES = [
    "transcripts_business",
    "transcripts_religion",
    "transcripts_comedy",
    "transcripts_lifestyle",
    "transcripts_tech",
    "transcripts_motivational",
    "transcripts_gaming",
    "transcripts_politics",
]

DEVANAGARI_RE = re.compile(r'[\u0900-\u097F]')
LATIN_RE = re.compile(r'[a-zA-Z]')

# thresholds — tune these once you see the distribution; YouTube videos
# run much shorter than the base paper's 10-min podcast cutoff
MIN_DURATION_SECONDS = 60      # was 10 minutes (600s) in base paper — adjust as needed
MIN_WORD_COUNT = 10

  # os.makedirs("./csv", exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
os.makedirs("./csv", exist_ok=True)

##Cell 2 — Helper functions (Unicode-safe, no ASCII stripping)

In [ ]:
def get_unicode_text(dictionary):
    """
    Concatenate all segment text while preserving Unicode characters
    and ensuring proper spacing between segments.
    """
    segments = [
        seg.get("text", "").strip()
        for seg in dictionary.get("segments", [])
        if seg.get("text", "").strip()
    ]
    return " ".join(segments)

def get_other_script_word_ratio(text):
    """
    Fraction of transcript words containing at least one letter
    outside Devanagari or Latin/Roman script.

    A word is counted as an 'other-script word' if it contains
    at least one alphabetic character that is neither:
      - Devanagari
      - Latin/Roman
    """

    words = text.split()

    if not words:
        return 0.0

    # Keep only words that contain at least one letter/number
    words = [
        w for w in words
        if any(c.isalnum() for c in w)
    ]

    if not words:
        return 0.0

    other_script_words = 0

    for word in words:

        # Extract Unicode alphabetic characters
        letters = [
            c for c in word
            if c.isalpha()
        ]

        if not letters:
            continue

        has_other_script = any(
            not (
                # Devanagari
                ('\u0900' <= c <= '\u097F')
                or
                # Latin / Roman
                ('A' <= c <= 'Z')
                or
                ('a' <= c <= 'z')
            )
            for c in letters
        )

        if has_other_script:
            other_script_words += 1

    return other_script_words / len(words)

def get_duration(dictionary):
    """
    Approximate duration from segment timestamps, since there's no
    top-level duration field in the JSON (unlike Spotify metadata).
    """
    segments = dictionary.get("segments", [])
    if not segments:
        return 0.0
    return float(segments[-1].get("end", 0.0))

def classify_script(text, whisperx_lang, dev_thresh=0.9, latin_thresh=0.9,
                    mix_floor=0.05, other_dominant=0.5,
                    other_threshold=0.05):
    """
    Returns:
        script_class, hindi_pct, english_pct, other_pct

    Script classes:
        - "other"       : other script is >=50% of alphabetic characters
        - "other_5_pct" : other script is >5% but <50%
        - "hindi"
        - "english"
        - "hindi-english (code-mixed)"
    """

    if whisperx_lang == "nn":
        return "noise", 0.0, 0.0, 0.0

    devanagari_chars = sum(
        1 for c in text
        if '\u0900' <= c <= '\u097F'
    )

    latin_chars = sum(
        1 for c in text
        if c.isascii() and c.isalpha()
    )

    other_chars = sum(
        1 for c in text
        if c.isalpha()
        and not c.isascii()
        and not ('\u0900' <= c <= '\u097F')
    )

    total = devanagari_chars + latin_chars + other_chars

    if total == 0:
        return "unknown/no_text", 0.0, 0.0, 0.0

    dev_pct = round(devanagari_chars / total * 100, 2)
    latin_pct = round(latin_chars / total * 100, 2)
    other_pct = round(other_chars / total * 100, 2)

    # ------------------------------------------
    # OTHER-SCRIPT CLASSIFICATION
    # ------------------------------------------

    # Original EDA definition:
    # Other script is dominant (>=50%)
    if other_pct / 100 >= other_dominant:
        script_class = "other"

    # Dataset-cleaning definition:
    # Other script is >5% but not dominant
    elif other_pct / 100 > other_threshold:
        script_class = "other_5_pct"

    # ------------------------------------------
    # NORMAL SCRIPT CLASSIFICATION
    # ------------------------------------------

    elif dev_pct / 100 >= dev_thresh:
        script_class = "hindi"

    elif latin_pct / 100 >= latin_thresh:
        script_class = "english"

    elif (
        dev_pct / 100 >= mix_floor
        and latin_pct / 100 >= mix_floor
    ):
        script_class = "hindi-english (code-mixed)"

    else:
        script_class = "other"

    return script_class, dev_pct, latin_pct, other_pct



def word_count(text):
    """
    Whitespace-based count works for both Devanagari and Latin script
    (Hindi uses spaces between words same as English), but strip
    punctuation-only tokens first so they don't inflate the count.
    """
    if not text:
        return 0
    tokens = [t for t in text.split() if any(c.isalnum() for c in t)]
    return len(tokens)

##Cell 3 — Walk all 8 communities and build the main df (slow — run once)

In [ ]:
records = []
error_files = []

for community in COMMUNITIES:
    folder_path = os.path.join(BASE_PATH, community)
    if not os.path.isdir(folder_path):
        print(f"WARNING: folder not found, skipping: {folder_path}")
        continue

    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file.endswith(".json"):
                full_path = os.path.join(root, file)
                try:
                    with open(full_path, encoding="utf-8") as f:
                        data = json.loads(f.read())

                    video_id = data.get("filename", file)
                    whisperx_lang = data.get("language", "unknown")
                    text = get_unicode_text(data)
                    duration = get_duration(data)
                    script_class, hindi_pct, english_pct, other_pct = classify_script(
                    text, whisperx_lang
                    )

                    n_words = word_count(text)

                    record = {
                    "video_id": video_id,
                    "community": community,
                    "whisperx_language": whisperx_lang,
                    "script_class": script_class,
                    "hindi_pct": hindi_pct,
                    "other_pct": other_pct,
                    "english_pct": english_pct,
                    "transcript": text,                      # only in get_dfs.ipynb
                    "transcript_length": n_words,            # only in get_dfs.ipynb
                    "duration": duration,                    # only in get_dfs.ipynb
                    "file_path": full_path,
                    }
                    records.append(record)

                except Exception as e:
                    error_files.append((full_path, str(e)))

df = pd.DataFrame(records)

print(f"Total JSON files processed: {len(records)}")
print(f"Files with errors: {len(error_files)}")
print(df.shape)
df.head()


Total JSON files processed: 9236
Files with errors: 0
(9236, 11)


,video_id,community,whisperx_language,script_class,hindi_pct,other_pct,english_pct,transcript,transcript_length,duration,file_path
0,AkshatZayn____G9-p4ImUU.mp3,transcripts_business,en,english,0.0,0.0,100.0,"Hey guys, so the size of India's eyewear marke...",1898,599.097,/content/drive/MyDrive/Transcripts_CSS/transcr...
1,AkshatZayn___wEGQCyB588.mp3,transcripts_business,en,english,0.0,0.0,100.0,Just before President Trump posted about his s...,1920,599.857,/content/drive/MyDrive/Transcripts_CSS/transcr...
2,AkshatZayn___zraNdztLIM.mp3,transcripts_business,en,english,0.0,0.0,100.0,"Hey guys, what's up? So on this video, I'm goi...",1897,597.054,/content/drive/MyDrive/Transcripts_CSS/transcr...
3,AkshatZayn__a8O7hupSrm0.mp3,transcripts_business,en,english,0.0,0.0,100.0,"Hey guys, what's up? So on this video, I'm goi...",2040,599.657,/content/drive/MyDrive/Transcripts_CSS/transcr...
4,AkshatZayn__aLJ1RL-9f7w.mp3,transcripts_business,en,english,0.0,0.0,100.0,"Hi everyone, so on this video I am going to he...",2135,600.038,/content/drive/MyDrive/Transcripts_CSS/transcr...


## FLAG AND DELETE FILES WITH >5% OTHER SCRIPT


In [ ]:
# ============================================================
# FLAG AND DELETE FILES WITH >5% OTHER SCRIPT
# ============================================================

other_5_pct_df = df[
    df["script_class"] == "other_5_pct"
].copy()

print(f"Files flagged as other_5_pct: {len(other_5_pct_df)}")

# Save a record of these files before deleting them
other_5_pct_df.to_csv(
    os.path.join(OUTPUT_DIR, "other_5_pct_df_new.csv"),
    index=False
)

# Delete the corresponding JSON files
deleted = 0
failed_delete = []

for _, row in other_5_pct_df.iterrows():
    file_path = row["file_path"]

    try:
        if os.path.exists(file_path):
            os.remove(file_path)
            deleted += 1
        else:
            failed_delete.append(
                (file_path, "file does not exist")
            )
    except Exception as e:
        failed_delete.append(
            (file_path, str(e))
        )

print(f"Deleted {deleted} JSON files.")
print(f"Failed to delete: {len(failed_delete)}")

# Remove these rows from the dataframe
df = df[
    df["script_class"] != "other_5_pct"
].copy()

print(f"Remaining rows after other_5_pct removal: {len(df)}")


Files flagged as other_5_pct: 0
Deleted 0 JSON files.
Failed to delete: 0
Remaining rows after other_5_pct removal: 9236


##Cell 4 — Filter to English / Hindi / code-mixed only

In [ ]:
KEEP_CLASSES = [
    "english",
    "hindi",
    "hindi-english (code-mixed)"
]

before = len(df)

df = df[df["script_class"].isin(KEEP_CLASSES)].copy()

print(f"Dropped {before - len(df)} rows "
    f"(other/unknown language) — kept {len(df)}")


Dropped 0 rows (other/unknown language) — kept 9236


##Cell 5 — Duration filter

In [ ]:
before = len(df)
too_short_df = df[df["duration"] < MIN_DURATION_SECONDS]
too_short_df.to_csv("./csv/too_short_df_new.csv", header=True, index=False)

df = df[df["duration"] >= MIN_DURATION_SECONDS]
print(f"Dropped {before - len(df)} rows under {MIN_DURATION_SECONDS}s duration — kept {len(df)}")

Dropped 22 rows under 60s duration — kept 9214


##Cell 6 — Word count filter (mirrors base paper's zero-word / low-word handling)

In [ ]:
zero_words_df = df[df["transcript_length"] == 0]
zero_words_df.to_csv("./csv/zero_words_df_new.csv", header=True, index=False)
df = df[df["transcript_length"] != 0]

less_than_min_words_df = df[df["transcript_length"] < MIN_WORD_COUNT]
less_than_min_words_df.to_csv("./csv/less_than_min_words_df_new.csv", header=True, index=False)
df = df[df["transcript_length"] >= MIN_WORD_COUNT]

print(f"Final row count after all filters: {len(df)}")

Final row count after all filters: 9206


##Cell 7 — Save final df

In [ ]:
#duration filter
too_short_df.to_csv(os.path.join(OUTPUT_DIR, "too_short_df_new.csv"), header=True, index=False)

#word count filter
zero_words_df.to_csv(os.path.join(OUTPUT_DIR, "zero_words_df_new.csv"), header=True, index=False)
less_than_min_words_df.to_csv(os.path.join(OUTPUT_DIR, "less_than_min_words_df_new.csv"), header=True, index=False)

#final df save
df.to_csv(os.path.join(OUTPUT_DIR, "df_new.csv"), header=True, index=False)
print(f"Saved {os.path.join(OUTPUT_DIR, 'df_new.csv')}")

Saved /content/drive/MyDrive/Transcripts_CSS/outputs/df_new.csv


In [ ]:
#parallel split for  faster computation

def split_dataframe(big_df, n_parts):
    indices = np.array(big_df.index)
    parts_indices = np.array_split(indices, n_parts)
    return [big_df.loc[idx] for idx in parts_indices]

big_df = pd.read_csv(os.path.join(OUTPUT_DIR, "df_new.csv"))

for n_parts, prefix in [(2, "df"), (4, "df-4")]:
    split_dfs = split_dataframe(big_df, n_parts)
    lens = []
    for i, part_df in enumerate(split_dfs):
        lens.append(len(part_df))
        part_df.to_csv(os.path.join(OUTPUT_DIR, f"{prefix}-{i}_new.csv"), header=True, index=False)
    assert sum(lens) == len(big_df)
    print(f"Split into {n_parts} parts: {lens}")

Split into 2 parts: [4603, 4603]
Split into 4 parts: [2302, 2302, 2301, 2301]


In [ ]:
#community based split for  faster computation
import os
import pandas as pd

big_df = pd.read_csv(os.path.join(OUTPUT_DIR, "df_new.csv"))

for community, community_df in big_df.groupby("community"):
    # Clean community name for use as a filename
    community_name = str(community).strip().lower().replace(" ", "_")

    output_path = os.path.join(
        OUTPUT_DIR,
        f"df_{community_name}_new.csv"
    )

    community_df.to_csv(
        output_path,
        header=True,
        index=False
    )

    print(
        f"{output_path}: "
        f"{len(community_df):,} rows"
    )

print(f"\nCreated {big_df['community'].nunique()} community files.")


/content/drive/MyDrive/Transcripts_CSS/outputs/df_transcripts_business_new.csv: 1,043 rows
/content/drive/MyDrive/Transcripts_CSS/outputs/df_transcripts_comedy_new.csv: 1,063 rows
/content/drive/MyDrive/Transcripts_CSS/outputs/df_transcripts_gaming_new.csv: 1,157 rows
/content/drive/MyDrive/Transcripts_CSS/outputs/df_transcripts_lifestyle_new.csv: 1,113 rows
/content/drive/MyDrive/Transcripts_CSS/outputs/df_transcripts_motivational_new.csv: 1,006 rows
/content/drive/MyDrive/Transcripts_CSS/outputs/df_transcripts_politics_new.csv: 1,357 rows
/content/drive/MyDrive/Transcripts_CSS/outputs/df_transcripts_religion_new.csv: 1,033 rows
/content/drive/MyDrive/Transcripts_CSS/outputs/df_transcripts_tech_new.csv: 1,434 rows

Created 8 community files.
